# Problema X11B10T0 — Solução via Funções de Green

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/x11b10t0.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*  
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Descrição

Placa plana de espessura $L$, com:

- **Face $x=0$:** temperatura prescrita $T_a$ (tipo 1) — *B10*: dígito 1.
- **Face $x=L$:** temperatura nula (tipo 1) — *B10*: dígito 0.
- **Condição inicial:** $T(x,0) = 0$ — *T0*.

Nomenclatura segundo o sistema de Beck: **X11B10T0**.

Este *notebook* avalia computacionalmente a solução analítica deduzida no §3.1 do livro.

## Solução analítica (§3.1 do livro)

A aplicação da equação solução em termos da função de Green $G_{X11}$ leva à série temporal

$$T(x,t) = -\frac{2 T_a}{\pi}\sum_{n=1}^{\infty}\frac{1}{n}\sin\!\left(\frac{n\pi x}{L}\right)\left(1 - e^{-n^{2}\pi^{2}\alpha t/L^{2}}\right),$$

que admite a **decomposição estacionária + transiente**:

$$\boxed{\;T(x,t) = T_a\left(1 - \frac{x}{L}\right) \;-\; \frac{2 T_a}{\pi}\sum_{n=1}^{\infty}\frac{1}{n}\sin\!\left(\frac{n\pi x}{L}\right)\,e^{-n^{2}\pi^{2}\alpha t/L^{2}}.\;}$$

O primeiro termo é o perfil linear de regime permanente (forma fechada exata); o segundo é a série transiente que decai exponencialmente.

Os autovalores aparecem em **forma fechada** $\beta_n = n\pi/L$ — raízes diretas de $\sin(\beta_n L) = 0$. Detalhes da construção, verificação da ortogonalidade e convergência da série de $G_{X11}$ estão em `gx11_autovalores.ipynb`.

## Bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (7.0, 4.2),
    'font.size':       11,
    'axes.grid':       True,
    'grid.alpha':      0.3,
})

## Parâmetros do problema

Valores típicos de uma amostra fina (mesma escala física do `x22b10t0.ipynb` para facilitar comparação).

In [ ]:
L     = 66e-4        # espessura da placa [m]   (66e-4 = 0,0066 m)
alpha = 1.93e-7      # difusividade térmica [m²/s]
Ta    = 100.0        # temperatura prescrita em x = 0 [°C]

M     = 200          # número de modos da série transiente

## Autovalores em forma fechada e vetores auxiliares

Não há equação transcendental: $\beta_n = n\pi/L$ é direto.

In [ ]:
n_arr = np.arange(1, M + 1)            # índices n = 1, 2, ..., M
beta  = np.pi * n_arr / L              # β_n = nπ/L
A     = alpha * beta**2                # A_n = α β_n²   (taxa de decaimento)

print(f'β_1 = {beta[0]:.4e} 1/m   →  ξ_1 = β_1·L = {beta[0]*L:.6f} (= π)')
print(f'A_1 = {A[0]:.4e} 1/s     →  1/A_1 = {1/A[0]:.4e} s')

## Funções: regime permanente, transiente e solução completa

Separamos as três para evidenciar a estrutura da Eq. boxed.

In [ ]:
def T_estac(x):
    """Regime permanente: forma fechada T_a (1 - x/L)."""
    return Ta * (1.0 - x/L)

def T_transiente(x, t_vec):
    """Série transiente: -(2 T_a/π) Σ (1/n) sin(nπx/L) exp(-n²π² α t / L²),
    avaliada num escalar x e num vetor de tempos t_vec.
    Retorna vetor do mesmo tamanho de t_vec."""
    sen_n  = np.sin(beta * x)                                # (M,)
    exp_nt = np.exp(-np.outer(t_vec, A))                      # (Nt, M)
    coef   = (1.0 / n_arr) * sen_n                            # (M,)
    return -(2.0 * Ta / np.pi) * (exp_nt * coef).sum(axis=1)  # (Nt,)

def T(x, t_vec):
    """Solução completa T(x, t) = T_estac(x) + T_transiente(x, t)."""
    return T_estac(x) + T_transiente(x, t_vec)

## Gráfico 1 — Temperatura ao longo do tempo em posições fixas

Acompanhamos a evolução temporal em três pontos da placa: próximo à face quente, no meio e próximo à face fria. As curvas devem convergir, em $t \to \infty$, para os valores estacionários $T_a(1-x/L)$.

In [ ]:
t = np.linspace(0, 600, 601)         # 0 a 10 min, passo 1 s

x_vals  = [0.1*L, 0.5*L, 0.9*L]
nomes   = [fr'$x = 0{{,}}1\,L$', fr'$x = 0{{,}}5\,L$', fr'$x = 0{{,}}9\,L$']
estilos = ['-', '--', '-.']

fig, ax = plt.subplots()
for x, nome, est in zip(x_vals, nomes, estilos):
    ax.plot(t, T(x, t), est, lw=1.8, label=nome)
    ax.axhline(T_estac(x), color='gray', ls=':', lw=0.8)
ax.set_xlabel('tempo [s]')
ax.set_ylabel('$T(x,t)$ [°C]')
ax.set_title(r'X11B10T0 — $T_a = 100$°C  em  $x=0$,  $T=0$  em  $x=L$')
ax.legend(loc='center right', fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.show()

## Gráfico 2 — Perfil espacial em vários instantes

Mostra a transição do perfil inicial $T(x,0) = 0$ para o perfil linear de regime permanente $T_a(1 - x/L)$ (traço cinza pontilhado).

In [ ]:
x = np.linspace(0, L, 401)
t_inst = [1.0, 10.0, 60.0, 300.0]
estilos = ['-', '--', '-.', ':']

fig, ax = plt.subplots()
# regime permanente (forma fechada)
ax.plot(x/L, T_estac(x), color='gray', lw=1.4,
        label=r'$T_{\mathrm{est}}(x) = T_a(1-x/L)$ — forma fechada')
# perfis transitórios
for tt, est in zip(t_inst, estilos):
    perfil = np.array([T(xi, np.array([tt]))[0] for xi in x])
    ax.plot(x/L, perfil, est, lw=1.8, label=fr'$t = {tt:g}$ s')
ax.set_xlabel(r'$x/L$')
ax.set_ylabel(r'$T(x,t)$ [°C]')
ax.set_title(r'Perfil espacial — X11B10T0')
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()

## Convergência da série transiente

A Eq. boxed escreve a solução como **forma fechada** $T_a(1-x/L)$ menos uma **série transiente** rapidamente convergente. Para $t > 0$ moderado, poucos modos bastam — o fator $e^{-n^2\pi^2\alpha t/L^2}$ amortece os modos altos com vigor.

In [ ]:
def T_truncado(x, t_vec, N):
    """Solução com a série transiente truncada nos N primeiros modos."""
    n_local    = np.arange(1, N + 1)
    beta_local = np.pi * n_local / L
    A_local    = alpha * beta_local**2
    sen_n      = np.sin(beta_local * x)
    exp_nt     = np.exp(-np.outer(t_vec, A_local))
    coef       = (1.0 / n_local) * sen_n
    return T_estac(x) - (2.0 * Ta / np.pi) * (exp_nt * coef).sum(axis=1)

x_eval     = 0.5 * L
t_vals     = np.array([1.0, 10.0, 100.0])
N_list     = [1, 2, 3, 5, 10, 20, 50]

import pandas as pd
linhas = []
for t_val in t_vals:
    T_full = T(x_eval, np.array([t_val]))[0]
    erros  = [abs(T_truncado(x_eval, np.array([t_val]), N)[0] - T_full) for N in N_list]
    linhas.append([t_val, T_full] + erros)

cols = ['t [s]', 'T(L/2, t) [°C]'] + [f'|erro| N={N}' for N in N_list]
df = pd.DataFrame(linhas, columns=cols).set_index('t [s]')
pd.set_option('display.float_format', '{:.3e}'.format)
print(df.to_string())

## Observações

- O perfil estacionário $T_a(1 - x/L)$ é uma **forma fechada exata** — não é aproximação da série de Fourier, é o valor que ela soma identicamente em $0 < x < L$.
- A série transiente converge extraordinariamente rápido (decaimento $e^{-n^2 \pi^2 \alpha t/L^2}$): a partir de $t \gtrsim L^2/(\pi^2\alpha)$ — escala difusiva característica — bastam 2 ou 3 modos para tolerância $10^{-3}$°C.
- Para tempos muito pequenos ($t \ll L^2/\alpha$) a série modal exige muitos termos; nesse regime é mais econômica a forma equivalente por imagens, explorada em `gx11_formas_fechadas.ipynb`.